# Read V1DD release-ingest data

Read the V1DD datasets plus the dedicated spatial-location, embedding, and connectome tables introduced by the release-ingest schema.

In [13]:
import polars as pl
from IPython.display import display

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    DatasetReader,
    read_cell_cell_connectivity,
    read_embedding_locations,
    read_embedding_spaces,
    read_reference_spaces,
    read_spatial_locations,
)
from connects_common_connectivity.models import LocationType

In [3]:
DATASET_ROOT = get_settings().output_root
PROJECT_ID = "v1dd_1196"
AXON_DATASET = "v1dd_1196_proofread_axons"
DENDRITE_DATASET = "v1dd_1196_proofread_dendrites"
HIERARCHY_ID = "EM_csm_cell_types"
CONNECTOME_ID = "v1dd_1196_em_proofread_axon_to_dendrite"
SYNAPSE_TABLE_ID = "v1dd_1196_synapses"
QC_EMBEDDING_ID = "v1dd_r84_qc_nonnull_umap"
AD_REF_EMBEDDING_ID = "v1dd_r84_ad_ref_umap"

reader = DatasetReader(DATASET_ROOT)

## Available datasets

In [14]:
reader.display_dataset_names()

id,name,project_id,modality
str,str,str,str
"""v1dd_1196_all""","""V1DD release 1196 — EM somas""","""v1dd_1196""","""ELECTRON_MICROSCOPY"""
"""v1dd_1196_proofread_axons""","""V1DD release 1196 — proofread …","""v1dd_1196""","""ELECTRON_MICROSCOPY"""
"""v1dd_1196_proofread_dendrites""","""V1DD release 1196 — proofread …","""v1dd_1196""","""ELECTRON_MICROSCOPY"""


## Related feature sets and cluster sets

In [5]:
for dataset_name in (AXON_DATASET, DENDRITE_DATASET):
    print(dataset_name)
    display(reader.display_featuresets(dataset_name))
    display(reader.display_clustersets(dataset_name))
    print('\n\n\n\n')

v1dd_1196_proofread_axons


feature_set_id,matrix_id,description,extraction_method,feature_definition_ids,cell_index_column
str,str,str,str,list[str],str
"""v1dd_soma_features""","""v1dd_1196_v1dd_soma_features""","""V1DD soma morphology and nucle…","""Loaded from v1dd_v1196 soma_fe…","[""nucleus_volume_um"", ""nucleus_area_um"", … ""soma_synapse_density_um""]","""id"""
"""v1dd_soma_volume""","""v1dd_1196_v1dd_soma_volume""","""V1DD release 1196 soma nucleus…","""Queried from the CAVE nucleus …","[""soma_volume""]","""id"""


clusterset_id,run,root,clusters
str,str,str,list[str]
"""EM_csm_cell_types""","""v1dd_cell_type_annotation_r84""","""neuron""","[""neuron"", ""exc"", … ""NeurogliaformFamily:NeurogliaformFamily-j""]"







v1dd_1196_proofread_dendrites


feature_set_id,matrix_id,description,extraction_method,feature_definition_ids,cell_index_column
str,str,str,str,list[str],str
"""v1dd_soma_features""","""v1dd_1196_v1dd_soma_features""","""V1DD soma morphology and nucle…","""Loaded from v1dd_v1196 soma_fe…","[""nucleus_volume_um"", ""nucleus_area_um"", … ""soma_synapse_density_um""]","""id"""
"""v1dd_soma_volume""","""v1dd_1196_v1dd_soma_volume""","""V1DD release 1196 soma nucleus…","""Queried from the CAVE nucleus …","[""soma_volume""]","""id"""


clusterset_id,run,root,clusters
str,str,str,list[str]
"""EM_csm_cell_types""","""v1dd_cell_type_annotation_r84""","""neuron""","[""neuron"", ""exc"", … ""NeurogliaformFamily:NeurogliaformFamily-j""]"


## Read dataset features and r84 cluster memberships

In [6]:
proofread_axons = reader.read_dataset(AXON_DATASET)
proofread_dendrites = reader.read_dataset(DENDRITE_DATASET)

print("proofread axons:", proofread_axons.shape)
display(proofread_axons.head())

print("proofread dendrites:", proofread_dendrites.shape)
display(proofread_dendrites.head())

proofread axons: (1164, 17)


dataitem_id,nucleus_volume_um,nucleus_area_um,nuclear_area_to_volume_ratio,nuclear_folding_area_um,fraction_nuclear_folding,nucleus_to_soma_ratio,soma_volume_um,soma_area_um,soma_to_nucleus_center_dist,soma_area_to_volume_ratio,soma_synapse_density_um,soma_volume,EM_csm_cell_types_level_0,EM_csm_cell_types_level_1,EM_csm_cell_types_level_2,EM_csm_cell_types_level_3
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,str,str,str,str
"""864691132534275418""",224.539952,228.434194,1.278639,19.476042,0.085259,0.456255,492.136677,612.704786,0.957844,2.03256,0.040803,224.683212,"""neuron""","""exc""","""L4""","""L4:L4-b"""
"""864691132534315610""",236.506655,246.509254,1.332867,27.856331,0.113003,0.422036,560.394755,691.239539,1.293702,2.102883,0.088247,236.639908,"""neuron""","""exc""","""L4""","""L4:L4-d"""
"""864691132535664474""",300.20032,264.670537,1.220714,3.614023,0.013655,0.397125,755.934198,923.133835,0.524689,2.300336,0.080162,300.366089,"""neuron""","""exc""","""L6CT""","""L6CT:L6CT-a"""
"""864691132536286810""",278.67367,330.243349,1.600612,79.708442,0.241363,0.38174,730.008386,707.65966,1.442203,1.804908,0.240228,278.909668,"""neuron""","""inh""","""MartinottiFamily""","""MartinottiFamily:MartinottiFam…"
"""864691132536904794""",null,null,null,null,null,null,null,null,null,null,null,273.808777,"""neuron""","""inh""","""MartinottiFamily""","""MartinottiFamily:MartinottiFam…"


proofread dendrites: (63986, 17)


dataitem_id,nucleus_volume_um,nucleus_area_um,nuclear_area_to_volume_ratio,nuclear_folding_area_um,fraction_nuclear_folding,nucleus_to_soma_ratio,soma_volume_um,soma_area_um,soma_to_nucleus_center_dist,soma_area_to_volume_ratio,soma_synapse_density_um,soma_volume,EM_csm_cell_types_level_0,EM_csm_cell_types_level_1,EM_csm_cell_types_level_2,EM_csm_cell_types_level_3
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,str,str,str,str
"""864691132496108732""",153.714823,262.44489,1.891226,87.028684,0.331607,0.52135,294.840211,292.666366,1.613731,1.366147,0.068337,153.827026,null,null,null,null
"""864691132511800666""",257.461754,283.770235,1.44991,42.661132,0.150337,0.968305,265.88904,281.628224,0.120611,1.408397,0.014203,257.700134,null,null,null,null
"""864691132525163794""",258.152734,266.504962,1.359263,38.271973,0.143607,0.427513,603.847392,767.669205,0.686228,2.221972,0.087277,258.204437,"""neuron""","""exc""","""L4""","""L4:L4-b"""
"""864691132533275738""",226.831331,231.126811,1.284983,26.172376,0.113238,0.445172,509.535857,800.333642,0.927358,2.594202,0.074969,226.991959,"""neuron""","""exc""","""L4""","""L4:L4-b"""
"""864691132533347418""",217.523027,230.473361,1.317649,33.566288,0.145641,0.460984,471.867091,688.681925,1.015647,2.349568,0.097287,217.654556,"""neuron""","""exc""","""L4""","""L4:L4-b"""


## Select only features or only clusters

In [7]:
axon_features_only = reader.read_dataset(
    AXON_DATASET,
    featuresets="v1dd_soma_features",
    clustersets=[],
)
axon_clusters_only = reader.read_dataset(
    AXON_DATASET,
    featuresets=[],
    clustersets=HIERARCHY_ID,
)

display(axon_features_only.head())
display(axon_clusters_only.head())

dataitem_id,nucleus_volume_um,nucleus_area_um,nuclear_area_to_volume_ratio,nuclear_folding_area_um,fraction_nuclear_folding,nucleus_to_soma_ratio,soma_volume_um,soma_area_um,soma_to_nucleus_center_dist,soma_area_to_volume_ratio,soma_synapse_density_um
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""864691132534275418""",224.539952,228.434194,1.278639,19.476042,0.085259,0.456255,492.136677,612.704786,0.957844,2.03256,0.040803
"""864691132534315610""",236.506655,246.509254,1.332867,27.856331,0.113003,0.422036,560.394755,691.239539,1.293702,2.102883,0.088247
"""864691132535664474""",300.20032,264.670537,1.220714,3.614023,0.013655,0.397125,755.934198,923.133835,0.524689,2.300336,0.080162
"""864691132536286810""",278.67367,330.243349,1.600612,79.708442,0.241363,0.38174,730.008386,707.65966,1.442203,1.804908,0.240228
"""864691132536904794""",null,null,null,null,null,null,null,null,null,null,null


dataitem_id,EM_csm_cell_types_level_0,EM_csm_cell_types_level_1,EM_csm_cell_types_level_2,EM_csm_cell_types_level_3
str,str,str,str,str
"""864691132534275418""","""neuron""","""exc""","""L4""","""L4:L4-b"""
"""864691132534315610""","""neuron""","""exc""","""L4""","""L4:L4-d"""
"""864691132535664474""","""neuron""","""exc""","""L6CT""","""L6CT:L6CT-a"""
"""864691132536286810""","""neuron""","""inh""","""MartinottiFamily""","""MartinottiFamily:MartinottiFam…"
"""864691132536904794""","""neuron""","""inh""","""MartinottiFamily""","""MartinottiFamily:MartinottiFam…"


## Validate dataset-centric reads

In [8]:
cluster_columns = {
    f"{HIERARCHY_ID}_level_1",
    f"{HIERARCHY_ID}_level_2",
    f"{HIERARCHY_ID}_level_3",
}

assert proofread_axons.height > 0
assert proofread_dendrites.height > 0
assert proofread_axons["dataitem_id"].n_unique() == proofread_axons.height
assert proofread_dendrites["dataitem_id"].n_unique() == proofread_dendrites.height
assert cluster_columns <= set(proofread_axons.columns)
assert cluster_columns <= set(proofread_dendrites.columns)
assert proofread_axons.select(
    pl.col(f"{HIERARCHY_ID}_level_2").is_not_null().sum()
).item() > 0
assert proofread_dendrites.select(
    pl.col(f"{HIERARCHY_ID}_level_2").is_not_null().sum()
).item() > 0

print("DatasetReader V1DD proofread-cohort checks passed.")

DatasetReader V1DD proofread-cohort checks passed.


In [11]:
reference_spaces = read_reference_spaces(
    project_id=PROJECT_ID, output_root=DATASET_ROOT
)
soma_locations = read_spatial_locations(
    PROJECT_ID,
    location_types=LocationType.SOMA,
    output_root=DATASET_ROOT,
)
embedding_spaces = read_embedding_spaces(
    PROJECT_ID, output_root=DATASET_ROOT
)
qc_embedding = read_embedding_locations(
    PROJECT_ID,
    embedding_space_ids=QC_EMBEDDING_ID,
    output_root=DATASET_ROOT,
)
ad_ref_embedding = read_embedding_locations(
    PROJECT_ID,
    embedding_space_ids=AD_REF_EMBEDDING_ID,
    output_root=DATASET_ROOT,
)
connectivity = read_cell_cell_connectivity(
    PROJECT_ID,
    CONNECTOME_ID,
    synapse_table_id=SYNAPSE_TABLE_ID,
    output_root=DATASET_ROOT,
)

assert {"v1dd_cave_nm", "v1dd_streamline_nm"} <= set(
    reference_spaces["id"].to_list()
)
assert soma_locations.select(
    "dataitem_id", "reference_space", "location_type"
).unique().height == soma_locations.height
assert {QC_EMBEDDING_ID, AD_REF_EMBEDDING_ID} <= set(
    embedding_spaces["id"].to_list()
)
assert qc_embedding["dataitem_id"].n_unique() == qc_embedding.height
assert ad_ref_embedding["dataitem_id"].n_unique() == ad_ref_embedding.height
assert connectivity["id"].n_unique() == connectivity.height
assert connectivity["connectome_id"].unique().to_list() == [CONNECTOME_ID]

display(reference_spaces)
display(soma_locations.head())
display(embedding_spaces)
display(qc_embedding.head())
display(ad_ref_embedding.head())
display(connectivity.head())

project_id,id,name,description,unit,voxel_size,voxel_size_unit,default_2d_view,anatomical_orientation
str,str,str,str,str,list[f64],str,struct[2],struct[7]
"""v1dd_1196""","""v1dd_cave_nm""","""V1DD CAVE nanometer coordinate…","""Soma nucleus positions returne…","""NANOMETERS_LENGTH""",null,null,"{""PLUS_X"",""MINUS_Y""}","{null,null,null,null,null,null,""PLUS_Y""}"
"""v1dd_1196""","""v1dd_streamline_nm""","""V1DD streamline-transformed na…","""Soma nucleus positions transfo…","""NANOMETERS_LENGTH""",null,null,"{""PLUS_X"",""MINUS_Y""}","{null,null,null,null,null,null,""PLUS_Y""}"


project_id,dataitem_id,reference_space,location_type,description,x,y,z
str,str,str,str,str,f64,f64,f64
"""v1dd_1196""","""864691132654552792""","""v1dd_cave_nm""","""SOMA""","""Soma nucleus position from CAV…",624680.0,531094.0,283770.0
"""v1dd_1196""","""864691132654552792""","""v1dd_streamline_nm""","""SOMA""","""Streamline-transformed soma nu…",-248.956909,222.054907,16.022035
"""v1dd_1196""","""864691132773514104""","""v1dd_cave_nm""","""SOMA""","""Soma nucleus position from CAV…",894573.0,478559.0,163530.0
"""v1dd_1196""","""864691132773514104""","""v1dd_streamline_nm""","""SOMA""","""Streamline-transformed soma nu…",30.864259,135.367984,-105.097754
"""v1dd_1196""","""864691132774106773""","""v1dd_cave_nm""","""SOMA""","""Soma nucleus position from CAV…",1.729859e6,674111.0,781200.0


project_id,id,name,description,embedding_method,parameters_json,input_feature_set_id,input_features_description,creation_date
str,str,str,str,str,str,str,str,date
"""v1dd_1196""","""v1dd_r84_qc_nonnull_umap""","""V1DD r84 QC-nonnull UMAP""","""The qc_nonnull_embed0/1 coordi…","""UMAP""","""{""source_columns"": [""qc_nonnul…",null,"""QC-nonnull axon/dendrite featu…",null
"""v1dd_1196""","""v1dd_r84_ad_ref_umap""","""V1DD r84 axon-dendrite referen…","""The ad_ref_embed0/1 coordinate…","""UMAP""","""{""source_columns"": [""ad_ref_em…",null,"""Axon-dendrite reference featur…",null


project_id,dataitem_id,embedding_space,x,y
str,str,str,f64,f64
"""v1dd_1196""","""864691132831533732""","""v1dd_r84_qc_nonnull_umap""",0.345442,-1.792807
"""v1dd_1196""","""864691132949445450""","""v1dd_r84_qc_nonnull_umap""",6.055111,14.027972
"""v1dd_1196""","""864691132958976541""","""v1dd_r84_qc_nonnull_umap""",3.275342,9.773444
"""v1dd_1196""","""864691132683159492""","""v1dd_r84_qc_nonnull_umap""",-1.15514,-2.168531
"""v1dd_1196""","""864691132624495256""","""v1dd_r84_qc_nonnull_umap""",1.597967,3.562651


project_id,dataitem_id,embedding_space,x,y
str,str,str,f64,f64
"""v1dd_1196""","""864691132949445450""","""v1dd_r84_ad_ref_umap""",2.862342,13.20245
"""v1dd_1196""","""864691132683159492""","""v1dd_r84_ad_ref_umap""",-0.038844,-4.722876
"""v1dd_1196""","""864691132624495256""","""v1dd_r84_ad_ref_umap""",-2.0359,1.671162
"""v1dd_1196""","""864691132620003003""","""v1dd_r84_ad_ref_umap""",2.735242,14.169513
"""v1dd_1196""","""864691132801183590""","""v1dd_r84_ad_ref_umap""",1.184313,16.24984


project_id,id,description,connectome_id,synapse_table_id,presynaptic_cell,postsynaptic_cell,measurement_type,modality,value,unit
str,str,str,str,str,str,str,str,str,f64,str
"""v1dd_1196""","""v1dd_1196_v1dd_1196_em_proofre…",null,"""v1dd_1196_em_proofread_axon_to…","""v1dd_1196_synapses""","""864691132536286810""","""864691132734919083""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",30.0,"""COUNT"""
"""v1dd_1196""","""v1dd_1196_v1dd_1196_em_proofre…",null,"""v1dd_1196_em_proofread_axon_to…","""v1dd_1196_synapses""","""864691132572190492""","""864691132606767301""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",2.0,"""COUNT"""
"""v1dd_1196""","""v1dd_1196_v1dd_1196_em_proofre…",null,"""v1dd_1196_em_proofread_axon_to…","""v1dd_1196_synapses""","""864691132558474249""","""864691132640932240""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT"""
"""v1dd_1196""","""v1dd_1196_v1dd_1196_em_proofre…",null,"""v1dd_1196_em_proofread_axon_to…","""v1dd_1196_synapses""","""864691132572056604""","""864691132829783452""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT"""
"""v1dd_1196""","""v1dd_1196_v1dd_1196_em_proofre…",null,"""v1dd_1196_em_proofread_axon_to…","""v1dd_1196_synapses""","""864691132558474249""","""864691132786392972""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",4.0,"""COUNT"""
